### 静态运行时上下文
静态运行时上下文(Static runtime Context)表示不可变的数据，如用户元数据、工具和传递给应用程序的数据库连接对象。通常在运行开始时通过 invoke / stream 的 context 参数传递，此类数据在运行期间不会更改。

In [2]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

out_model = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

#### 1. 中间件中访问
1.1 Node-style hooks

In [3]:
from dataclasses import dataclass
from typing import Any
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, AgentState,hook_config
from langchain.messages import AIMessage
from langgraph.runtime import Runtime
from langgraph.store.memory import InMemoryStore
from dotenv import load_dotenv
from loguru import logger

load_dotenv()

store = InMemoryStore()

namespace = ("users", "credits")

key1 = 'Ada Lovelace'
value1 = {
    "tokens_credit_left": 5000,
    "user_level": 5
}

key2 = 'Blackwell'
value2 = {
    "tokens_credit_left": 2999,
    "user_level": 5
}

key3 = 'Ampere'
value3 = {
    "tokens_credit_left": 1000,
    "user_level": 5
}

store.put(namespace, key1, value1)
store.put(namespace, key2, value2)
store.put(namespace, key3, value3)

@dataclass
class UserContext:
    username: str

class CheckCredit(AgentMiddleware):

    @hook_config(can_jump_to=["end"])
    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        store = runtime.store
        context = runtime.context
        username = context.username
        credit = store.get(namespace, username)
        if not credit:
            return {
                "jump_to": "end",
                "messages": AIMessage("您尚未注册~")
             }
        if credit.value["tokens_credit_left"] < 3000:
            return {
                "jump_to": "end",
                "messages": AIMessage(f"{username}额度不足，请充值")
            }
        return None

    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str,Any] | None:
        store = runtime.store
        context = runtime.context
        username = context.username
        credit = store.get(namespace, username)
        usage_metadata = state["messages"][-1].usage_metadata
        token_usage = usage_metadata["input_tokens"] + usage_metadata["output_tokens"] * 6
        credit.value["tokens_credit_left"] -= token_usage
        # 更新长期记忆
        store.put(namespace, username, credit.value)
        return None

    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str,Any] | None:
        store = runtime.store
        context = runtime.context
        username = context.username
        item = store.get(namespace, username)
        credit = item.value["tokens_credit_left"]
        logger.info(f"{username} 当前剩余额度：{credit}")
        return None

agent = create_agent(
    model=out_model,
    middleware=[CheckCredit()],
    store=store,
    context_schema=UserContext
)

response1 = agent.invoke(
    {"messages": ["你好啊，你知道 Ada Lovelace 的贡献吗？"]},
    context=UserContext(username="Ada Lovelace")
)
print("=" * 30, '-> Ada Lovelace messages <-', '=' * 30)
for msg in response1["messages"]:
    msg.pretty_print()

response2 = agent.invoke(
    {"messages": ["你好啊，你知道 Blackwell 的贡献吗？"]},
    context=UserContext(username="Blackwell")
)
print("=" * 30, '-> Blackwell messages <-', '=' * 30)
for msg in response2["messages"]:
    msg.pretty_print()

2026-10-13 13:37:22.607 | INFO     | __main__:after_agent:81 - Ada Lovelace 当前剩余额度：-3728
2026-10-13 13:37:22.609 | INFO     | __main__:after_agent:81 - Blackwell 当前剩余额度：2999


============================== -> Ada Lovelace messages <- ==============================
================================ Human Message =================================

你好啊，你知道 Ada Lovelace 的贡献吗？
================================== Ai Message ==================================

你好！当然知道。阿达·洛夫莱斯（Ada Lovelace，全名奥古斯塔·埃达·金，洛夫莱斯伯爵夫人，1815–1852）是19世纪英国著名的数学家与作家，被国际公认为**世界上第一位计算机程序员**。她的贡献远超其所处时代，为现代计算机科学奠定了重要的思想基础。以下是她的核心贡献：

🔹 **《分析机注记》（Notes on the Analytical Engine）**  
1843年，她翻译并大幅扩充了意大利学者路易吉·梅纳比亚关于查尔斯·巴贝奇“分析机”（机械式通用计算机雏形）的论文。她在译文后追加了长达7页的“注记N”，篇幅几乎是原文的三倍，这份文件成为她最重要的传世之作。

🔹 **编写史上首个完整计算机算法**  
在注记G中，她详细描述了用分析机计算伯努利数的步骤。这不仅是一个数学计算流程，更是**人类历史上第一个为机器编写的、结构完整的算法程序**，包含循环、条件跳转等现代编程的基本逻辑。

🔹 **提出“通用计算”与程序控制的前瞻思想**  
巴贝奇最初将分析机视为高级计算器，但洛夫莱斯敏锐地指出：这台机器若能处理数字，理论上也能处理其他符号（如音符或字母）。她写道：“分析机编织代数模式，正如提花织机编织花朵与叶片。”这一洞见跳出了纯数值计算的框架，预见了计算机作为**通用信息处理工具**的可能性。

🔹 **区分“计算”与“智能”的边界**  
她明确强调分析机本身不具备创造力，它只能执行人类事先设计好的指令序列。这一观点奠定了现代程序设计的核心理念：**计算机是忠实的执行者，真正的逻辑与创造力来自人类的规划**。

🌍 **历史影响与现代纪念**  
尽管分析机因技术限制在她生前未能真正造出

1.2 Wrap-style hooks

① wrap_model_cal

In [4]:
from dataclasses import dataclass
from typing import Callable
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, ModelResponse,ModelRequest
from langchain.messages import AIMessage
from langchain.tools import tool
from langgraph.store.memory import InMemoryStore
from dotenv import load_dotenv
from loguru import logger
load_dotenv()
store = InMemoryStore()
namespace = ("users", "credits")

key1 = 'Ada Lovelace'
value1 = {
    "tokens_left": 3000, # 剩余大模型token额度
    "get_weather": "yes", # 是否认证天气查询服务
    "get_news": "no" # 是否认证网络爬虫服务，用于查询新闻
}

key2 = 'Blackwell'
value2 = {
    "tokens_left": 2000, # 剩余大模型token额度
    "get_weather": "no", # 是否认证天气查询服务
    "get_news": "yes" # 是否认证网络爬虫服务，用于查询新闻
}

key3 = 'Ampere'
value3 = {
    "tokens_left": 3000, # 剩余大模型token额度
    "get_weather": "yes", # 是否认证天气查询服务
    "get_news": "yes" # 是否认证网络爬虫服务，用于查询新闻
}

store.put(namespace, key1, value1)
store.put(namespace, key2, value2)
store.put(namespace, key3, value3)

@tool
def get_weather(city: str):
    """
    查询指定城市当日天气
    Args:
    city: 城市名称
    """
    return f"{city} 今天天气不错"

@tool
def get_news():
    """查询当日新闻"""
    return "美伊尚未达成停战协议"

@dataclass
class UserContext:
    username: str

class CheckCredit(AgentMiddleware):
    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
        ) -> ModelResponse:
            context = request.runtime.context
            store = request.runtime.store
            value = store.get(namespace, context.username).value
            tools = []
            for tool in request.tools:
                tool_name = tool.name
                if value[tool_name] == "yes":
                    tools.append(tool)
                else:
                    logger.warning(f"{context.username} 无权调用 {tool_name}")
            # 更改调用请求携带的工具集，仅本次生效
            request = request.override(tools=tools)
            return handler(request)

agent = create_agent(
    model=out_model,
    middleware=[CheckCredit()],
    tools=[get_weather, get_news],
    store=store,
    context_schema=UserContext
)

response1 = agent.invoke(
    {"messages": ["你好啊，今天北京天气如何，今日有哪些新闻？"]},
    context=UserContext(username="Ada Lovelace")
)
print("=" * 30, '-> Ada Lovelace messages <-', '=' * 30)
for msg in response1["messages"]:
    msg.pretty_print()

response2 = agent.invoke(
    {"messages": ["你好啊，今天北京天气如何，今日有哪些新闻？"]},
    context=UserContext(username="Blackwell")
)
print("=" * 30, '-> Blackwell messages <-', '=' * 30)
for msg in response2["messages"]:
    msg.pretty_print()

response3 = agent.invoke(
    {"messages": ["你好啊，今天北京天气如何，今日有哪些新闻？"]},
    context=UserContext(username="Ampere")
)
print("=" * 30, '-> Ampere messages <-', '=' * 30)
for msg in response3["messages"]:
    msg.pretty_print()

2026-10-13 13:44:51.163 | WARNING  | __main__:wrap_model_call:72 - Ada Lovelace 无权调用 get_news
2026-10-13 13:45:01.129 | WARNING  | __main__:wrap_model_call:72 - Ada Lovelace 无权调用 get_news
2026-10-13 13:45:08.150 | WARNING  | __main__:wrap_model_call:72 - Blackwell 无权调用 get_weather


============================== -> Ada Lovelace messages <- ==============================
================================ Human Message =================================

你好啊，今天北京天气如何，今日有哪些新闻？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_868fc81bdc5d4aa48b1fb7b3)
 Call ID: call_868fc81bdc5d4aa48b1fb7b3
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京 今天天气不错
================================== Ai Message ==================================

北京今天的天气不错。

至于您询问的今日新闻，由于我暂时没有获取实时新闻的工具，无法为您播报当天的具体新闻内容。建议您通过新闻APP、官方网站或社交媒体关注最新热点。


2026-10-13 13:45:15.563 | WARNING  | __main__:wrap_model_call:72 - Blackwell 无权调用 get_weather
2026-10-13 13:45:17.286 | WARNING  | __main__:wrap_model_call:72 - Blackwell 无权调用 get_weather


============================== -> Blackwell messages <- ==============================
================================ Human Message =================================

你好啊，今天北京天气如何，今日有哪些新闻？
================================== Ai Message ==================================
Tool Calls:
  get_news (call_5272b0b8806b4c76a47cfe69)
 Call ID: call_5272b0b8806b4c76a47cfe69
  Args:
================================= Tool Message =================================
Name: get_news

美伊尚未达成停战协议
================================== Ai Message ==================================
Tool Calls:
  get_news (call_d72e4a47d0e54296a130e5c2)
 Call ID: call_d72e4a47d0e54296a130e5c2
  Args:
================================= Tool Message =================================
Name: get_news

美伊尚未达成停战协议
================================== Ai Message ==================================

您好！很遗憾，由于我没有直接查询天气的工具，所以无法为您提供今天北京的天气信息。

关于您询问的今日新闻，我查阅到的资讯是：**美伊尚未达成停战协议**。
============================== -> Ampere messages <- ============

② wrap_tool_call

In [5]:
from dataclasses import dataclass
from typing import Callable, Any
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, ModelResponse,ModelRequest, ToolCallRequest
from langchain.messages import AIMessage, ToolMessage
from langchain.tools import tool
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command
from dotenv import load_dotenv
from loguru import logger

load_dotenv()
store = InMemoryStore()

namespace = ("users", "credits")

key1 = 'Ada Lovelace'
value1 = {
    "tokens_left": 3000, # 剩余大模型token额度
    "get_weather": "yes", # 是否认证天气查询服务
    "get_news": "no" # 是否认证网络爬虫服务，用于查询新闻
}

key2 = 'Blackwell'
value2 = {
    "tokens_left": 2000, # 剩余大模型token额度
    "get_weather": "no", # 是否认证天气查询服务
    "get_news": "yes" # 是否认证网络爬虫服务，用于查询新闻
}

key3 = 'Ampere'
value3 = {
    "tokens_left": 3000, # 剩余大模型token额度
    "get_weather": "yes", # 是否认证天气查询服务
    "get_news": "yes" # 是否认证网络爬虫服务，用于查询新闻
}

store.put(namespace, key1, value1)
store.put(namespace, key2, value2)
store.put(namespace, key3, value3)
# 工具额度消耗
CREDIT_MAP = {
    "get_weather": 10,
    "get_news": 20
}

@tool
def get_weather(city: str):
    """
    查询指定城市当日天气
    Args:
        city: 城市名称
    """
    return f"{city} 今天天气不错"

@tool
def get_news():
    """查询当日新闻"""
    return "美伊尚未达成停战协议"

@dataclass
class UserContext:
    username: str

class CheckCredit(AgentMiddleware):
    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
        ) -> ModelResponse:
            context = request.runtime.context
            store = request.runtime.store
            value = store.get(namespace, context.username).value
            tools = []
            for tool in request.tools:
                tool_name = tool.name
                if value[tool_name] == "yes":
                    tools.append(tool)
                else:
                    logger.warning(f"{context.username} 无权调用 {tool_name}")
            # 更改调用请求携带的工具集，仅本次生效
            request = request.override(tools=tools)
            return handler(request)

class ToolGuard(AgentMiddleware):
    def wrap_tool_call(
        self,
        request: ToolCallRequest,
        handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
    ) -> ToolMessage | Command[Any]:
        store = request.runtime.store
        context = request.runtime.context
        username = context.username
        value = store.get(namespace, username).value
        tool_name = request.tool.name
        # 本次调用花费
        credits_cost = CREDIT_MAP.get(tool_name)
        # 剩余额度
        credits_left = value["tokens_left"]
        if credits_left < credits_cost:
            logger.warning(f"{username} 额度不足，{tool_name} 调用失败！")
            return Command(
                update={
                    "messages": [
                        ToolMessage(
                            content="额度不足，无法调用工具，请充值~",
                            tool_call_id=request.runtime.tool_call_id
                        )
                    ]
                }
            )
        # 更新剩余额度
        credits_left -= credits_cost
        value["tokens_left"] = credits_left
        store.put(namespace, username, value)
        logger.info(f"{username} 调用 {tool_name} 花费 {credits_cost}，剩余额度 {credits_left}")
        logger.info(f"{username} 状态已更新，当前状态 {store.get(namespace,username).value}")
        return handler(request)

agent = create_agent(
    model=out_model,
    middleware=[CheckCredit(), ToolGuard()],
    tools=[get_weather, get_news],
    store=store,
    context_schema=UserContext
)

response = agent.invoke(
    {"messages": ["你好啊，今天北京天气如何，今日有哪些新闻？"]},
    context=UserContext(username="Ada Lovelace")
)
print("=" * 30, '-> Ada Lovelace messages <-', '=' * 30)
for msg in response["messages"]:
    msg.pretty_print()

2026-10-13 13:53:57.956 | WARNING  | __main__:wrap_model_call:80 - Ada Lovelace 无权调用 get_news
2026-10-13 13:54:04.433 | INFO     | __main__:wrap_tool_call:116 - Ada Lovelace 调用 get_weather 花费 10，剩余额度 2990
2026-10-13 13:54:04.436 | INFO     | __main__:wrap_tool_call:117 - Ada Lovelace 状态已更新，当前状态 {'tokens_left': 2990, 'get_weather': 'yes', 'get_news': 'no'}
2026-10-13 13:54:04.437 | WARNING  | __main__:wrap_model_call:80 - Ada Lovelace 无权调用 get_news


============================== -> Ada Lovelace messages <- ==============================
================================ Human Message =================================

你好啊，今天北京天气如何，今日有哪些新闻？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_77d0e8dee1eb48b793d44be4)
 Call ID: call_77d0e8dee1eb48b793d44be4
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京 今天天气不错
================================== Ai Message ==================================

您好！据查询，北京今天天气不错。不过，我无法提供今日的新闻资讯。


1.3 特殊的便捷中间件dynamic_prompt

@dynamic_prompt底层实现机制和通用钩子函数是统一的，对于动态提示词处理提供了更加便捷的入口。

In [6]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langgraph.store.memory import InMemoryStore
from dotenv import load_dotenv
from loguru import logger
load_dotenv()
# 准备历史记忆
namespace = ("users", "preferences")
key1 = "Ada Lovelace"
value1 = {
    "chat_preferences": [
    "不喜欢啰嗦",
    "尽可能用最精简的文字解释清楚"
    ]
}
key2 = "Blackwell"
value2 = {
    "chat_preferences": [
    "喜欢长篇大论、引经据典",
    "不知道的东西要明确说'不知道'，不能胡编乱造"
    ]
}

store = InMemoryStore()
store.put(namespace, key1, value1)
store.put(namespace, key2, value2)

@dataclass
class UserContext:
    username: str

@dynamic_prompt
def personalized_prompt(request: ModelRequest) -> str:
    username = request.runtime.context.username
    store = request.runtime.store
    preferences = store.get(namespace, username).value["chat_preferences"]
    custom_prompt = f"# 用户偏好\n{'\n'.join(preferences)}"
    logger.info(f"{username} 自定义系统提示词\n{custom_prompt}")
    return custom_prompt

agent = create_agent(
    model=out_model,
    middleware=[personalized_prompt],
    store=store,
    context_schema=UserContext
)

response1 = agent.invoke(
    {"messages": ["为什么花儿这样红？"]},
    context=UserContext(username="Ada Lovelace")
)
print("=" * 30, '-> Ada Lovelace messages <-', '=' * 30)
for msg in response1["messages"]:
    msg.pretty_print()

response2 = agent.invoke(
    {"messages": ["为什么花儿这样红？"]},
    context=UserContext(username="Blackwell")
)
print("=" * 30, '-> Blackwell messages <-', '=' * 30)
for msg in response2["messages"]:
    msg.pretty_print()

2026-10-13 14:14:34.566 | INFO     | __main__:personalized_prompt:39 - Ada Lovelace 自定义系统提示词
# 用户偏好
不喜欢啰嗦
尽可能用最精简的文字解释清楚
2026-10-13 14:14:41.129 | INFO     | __main__:personalized_prompt:39 - Blackwell 自定义系统提示词
# 用户偏好
喜欢长篇大论、引经据典
不知道的东西要明确说'不知道'，不能胡编乱造


============================== -> Ada Lovelace messages <- ==============================
================================ Human Message =================================

为什么花儿这样红？
================================== Ai Message ==================================

主要因含**花青素**。其色泽受细胞液酸碱度调控，进化目的为吸引昆虫传粉。
============================== -> Blackwell messages <- ==============================
================================ Human Message =================================

为什么花儿这样红？
================================== Ai Message ==================================

“何以解忧？唯有杜康。”然花之红，非关酒醴，乃天地生化之机、亿万年演运之选、诗人哲思之影所共铸也。欲穷其理，须自分子之微，涉于细胞之构，溯至生态之选，终归于人文之映。兹分四层陈之，以飨知音。若遇考据未周或学界存疑处，某必依嘱直言“不知”，绝不穿凿附会。

---

### 一、分子之微：花青素与显色化学的精密编排

花之红色，首要功臣为**花青素**（Anthocyanins），属黄酮类次生代谢产物，常见衍生物包括矢车菊素（cyanidin）、飞燕草素（delphinidin）与芍药素（peonidin）。其分子结构含多个酚羟基与糖苷基团，显色绝非单一色素之功，实为**“花青素-辅色素-金属离子-液泡pH”**四者协同之结果。

清代植物学家虽无现代光谱仪，然《群芳谱》《广群芳谱》屡载“海棠胭脂色，得朝露则愈艳”，今人据此推知：光照与水分影响液泡渗透势与pH，进而调控色素稳定性。现代植物生理学明示：花青素在酸性环境（pH≤5）中因氧鎓盐结构稳定而呈鲜红至紫红；中

#### 2. 工具中访问

In [11]:
from pydantic import BaseModel, Field
from typing import List, Any
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import HumanMessage, SystemMessage
from langgraph.prebuilt.tool_node import ToolRuntime
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from loguru import logger
from dotenv import load_dotenv

load_dotenv()

@dataclass
class UserContext:
    user_id: str

store = InMemoryStore()
namespace = ("users", "user_info")

class UserInfo(BaseModel):
    username: str = Field(description="用户名", default="unknown")
    age: int = Field(description="年龄", default=0)
    # 这里相当于告诉模型，每次创建新实例时，调用list()创建一个新的list传进去
    hobbies: List[str] = Field(description="兴趣爱好", default_factory=list)
    # 这种写法是用可变对象做默认值，会有多个对象共享一个列表对象的风险
    # hobbies: List[str] = Field(description="兴趣爱好", default=[])

@tool(parse_docstring=True)
def read_user_info(runtime: ToolRuntime[UserContext, Any]) -> UserInfo |str:
    """
    读取用户信息

    Args:
        runtime: 运行时间

    Returns:
        UserInfo | str: 如果找到用户信息，返回 UserInfo 对象；否则返回空字符串。
    """
    user_id = runtime.context.user_id
    store = runtime.store
    item = store.get(namespace, user_id)
    if item:
        return UserInfo(**item.value)
    return ''

@tool(parse_docstring=True)
def write_user_info(user_info: UserInfo, runtime: ToolRuntime[UserContext,Any]) -> bool:
    """
    将用户信息写入长期记忆

    Args:
        user_info: 用户信息对象
        runtime: 运行时间

    Returns:
        bool: 如果成功写入则返回True，否则返回False
    """
    user_id = runtime.context.user_id
    store = runtime.store
    try:
        logger.info(f"写入记忆, namespace: {namespace}, user_id: {user_id},user_info: {user_info.model_dump()}")
        store.put(namespace, user_id, user_info.model_dump())
    except Exception as e:
        logger.error(e)
        return False
    return True

agent = create_agent(
    model = out_model,
    tools = [read_user_info, write_user_info],
    store = store,
    checkpointer=InMemorySaver(),
    context_schema=UserContext
)

config1 = {"configurable": {"thread_id": "thread_1"}}
config2 = {"configurable": {"thread_id": "thread_2"}}
agent.invoke(
    {
        "messages": [
            SystemMessage("如果输入包含用户信息，抽取并调用工具记录。在记录之前先查找历史信息，和新增信息合并后记录。"
            "如果用户提问设计个人信息，调用工具查找"),
            HumanMessage("你好，我是韩立，我喜欢修仙")
        ]
    },
    context = UserContext(user_id = 'user_1'),
    config = config1
)

thread_1_response = agent.invoke(
    {"messages": ["我今年两百岁了，是你们口中的 “元婴老怪”，我喜欢跑步"]},
    context = UserContext(user_id = 'user_1'),
    config = config1
)
print("=" * 30, '-> thread_1 messages <-', '=' * 30)
for msg in thread_1_response["messages"]:
    msg.pretty_print()

thread_2_response = agent.invoke(
    {"messages": ["你还记得我吗？"]},
    context = UserContext(user_id = 'user_1'),
    config = config2
)
print("=" * 30, '-> thread_2 messages <-', '=' * 30)
for msg in thread_2_response["messages"]:
    msg.pretty_print()

2026-10-13 14:51:17.549 | INFO     | __main__:write_user_info:63 - 写入记忆, namespace: ('users', 'user_info'), user_id: user_1,user_info: {'username': '韩立', 'age': 0, 'hobbies': ['修仙']}
2026-10-13 14:51:28.816 | INFO     | __main__:write_user_info:63 - 写入记忆, namespace: ('users', 'user_info'), user_id: user_1,user_info: {'username': '韩立', 'age': 200, 'hobbies': ['修仙', '跑步']}


============================== -> thread_1 messages <- ==============================
================================ System Message ================================

如果输入包含用户信息，抽取并调用工具记录。在记录之前先查找历史信息，和新增信息合并后记录。如果用户提问设计个人信息，调用工具查找
================================ Human Message =================================

你好，我是韩立，我喜欢修仙
================================== Ai Message ==================================
Tool Calls:
  read_user_info (call_3366e3f1c1b4467f933f8060)
 Call ID: call_3366e3f1c1b4467f933f8060
  Args:
================================= Tool Message =================================
Name: read_user_info


================================== Ai Message ==================================
Tool Calls:
  write_user_info (call_14dd3b3a4f90420ca865460b)
 Call ID: call_14dd3b3a4f90420ca865460b
  Args:
    user_info: {'username': '韩立', 'hobbies': ['修仙']}
================================= Tool Message =================================
Name: write_user_info

true
========================

分析
1. 前两次invoke通过相同的thread_id串联为一个会话，共享短期记忆
2. context是运行时静态配置，每次invoke互相独立
3. 长期记忆可以在任意位置访问，全局共享同一份信息
4. tool参数列表中的ToolRuntime是Langchain / Langgraph运行时注入的，在工具schema中没有体
现
5. ToolRuntime的默认泛型是[None, Any]，第一个是Context的泛型，第二个是State对象的泛型。
如果不显式指定上下文对象的类型，则Langchain/Langgraph底层会认为上下文对象为None，
invoke调用时传递了context就会抛出警告，所以这里将UserContext作为ToolRuntime的第一个
泛型传递。